# qbraid.runtime.quantinuum

> **Billing notice:** the Quantinuum emulators on qBraid are currently **free to run** (0 credits). You need a qBraid API key, but no Quantinuum or NEXUS account.

[Quantinuum](https://www.quantinuum.com)'s System Model H1 and H2 are **trapped-ion** quantum computers: all-to-all qubit connectivity, high-fidelity gates, and native support for measuring and resetting qubits partway through a circuit. qBraid gives you four of Quantinuum's [NEXUS](https://nexus.quantinuum.com)-hosted emulators of those machines:

| Device ID | Qubits | Noise |
|---|---|---|
| `quantinuum:quantinuum:sim:h1-1le` | 20 | none (ideal statevector) |
| `quantinuum:quantinuum:sim:h2-1le` | 26 | none (ideal statevector) |
| `quantinuum:quantinuum:sim:h1-emulator` | 20 | H1 hardware noise model |
| `quantinuum:quantinuum:sim:h2-emulator` | 26 | H2 hardware noise model |

The **LE** targets are noiseless: the only randomness in the counts is shot noise, which makes them the place to check that a circuit is *correct*. The **Emulator** targets add Quantinuum's model of the real device's errors, so they show what a circuit would look like on hardware.

This notebook goes through the qBraid platform with `QbraidProvider`. If you have your own Quantinuum NEXUS credentials, `QuantinuumProvider` talks to NEXUS directly instead; see [`Provider-Jobs/qbraid_runtime_quantinuum_provider.ipynb`](../Provider-Jobs/qbraid_runtime_quantinuum_provider.ipynb).

It covers:

1. Connecting and running a first circuit
2. Inspecting the program Quantinuum's compiler actually ran
3. The `optimisation_level` runtime option
4. Noiseless vs. noisy emulation
5. Mid-circuit measurement and reset
6. Running several circuits

In [ ]:
%%capture

# "%%capture" hides the install output; comment it out if you need to debug this step.
%pip install 'qbraid[qiskit,visualization]'

## Connect to the device

`QbraidProvider` picks up your API key from `~/.qbraid/qbraidrc` or the `QBRAID_API_KEY`
environment variable. Inside qBraid Lab it is already configured for you.

In [ ]:
from qbraid.runtime import QbraidProvider

provider = QbraidProvider()
device = provider.get_device("quantinuum:quantinuum:sim:h2-1le")

device.metadata()

26 qubits, `ONLINE`, and zero pricing. The device takes OpenQASM 2, and qBraid transpiles to
it for you, so you can pass a Qiskit, Cirq, Braket, pyQuil, or pytket circuit straight to
`run()`:

In [ ]:
device.supported_run_inputs()

## A first run

A three-qubit GHZ state, written in Qiskit:

In [ ]:
from qiskit import QuantumCircuit


def ghz(num_qubits: int) -> QuantumCircuit:
    circuit = QuantumCircuit(num_qubits)
    circuit.h(0)
    for qubit in range(num_qubits - 1):
        circuit.cx(qubit, qubit + 1)
    circuit.measure_all()
    return circuit


circuit = ghz(3)
circuit.draw()

Every NEXUS job runs in stages: the circuit is uploaded, **compiled** for the target, and then
executed. Even a tiny circuit usually takes a minute or two from submission to result, so
wait with a timeout rather than blocking indefinitely:

In [ ]:
def get_counts(job, timeout: int = 600) -> dict[str, int]:
    job.wait_for_final_state(timeout=timeout, poll_interval=5)
    return job.result().data.get_counts()


job = device.run(circuit, shots=200)
get_counts(job)

Two peaks, `000` and `111`, as a GHZ state should give. Keys are little-endian: **qubit 0 is
the rightmost bit**.

## What actually ran

Before execution, NEXUS compiles the circuit into the trapped-ion **native gate set**:
single-qubit rotations `U1q` and `rz`, and a two-qubit `RZZ` (ZZ) interaction. Your `CX`
gates don't exist on the hardware. `compiled_program()` returns the program that was really
executed, as OpenQASM 2 in Quantinuum's `hqslib1` dialect:

In [ ]:
compiled = job.compiled_program()

print(compiled.format)
print(compiled.data)

The two `CX` gates became two `RZZ` gates plus single-qubit rotations around them.

## Runtime options

Settings go in a **`runtime_options` dict**:

```python
device.run(circuit, shots=200, runtime_options={"optimisation_level": 2})
```

The Quantinuum integration supports one option:

| Option | What it does |
|---|---|
| `optimisation_level` | How hard the NEXUS compiler works to simplify the circuit: an integer from `0` (rebase to native gates only) to `3`. Default `1` |

Anything else in `runtime_options` is **ignored**, without an error. In particular there is no
`seed`: sampling is never reproducible across jobs.

## Choosing an optimisation level

Here is a circuit with a lot of redundancy: two Hadamards that cancel, two `CX` gates that
cancel, and two `RZ` rotations that cancel. All that is left is a Bell pair.

In [ ]:
redundant = QuantumCircuit(2)
redundant.h(0)
redundant.h(0)
redundant.cx(0, 1)
redundant.cx(0, 1)
redundant.rz(0.3, 1)
redundant.rz(-0.3, 1)
redundant.h(0)
redundant.cx(0, 1)
redundant.measure_all()

redundant.draw()

Submit it at levels 0, 1 and 2. The jobs run in parallel on NEXUS, so submit all three
before waiting on any:

In [ ]:
opt_jobs = {
    level: device.run(redundant, shots=200, runtime_options={"optimisation_level": level})
    for level in (0, 1, 2)
}

for level, opt_job in opt_jobs.items():
    counts = get_counts(opt_job)
    two_qubit_gates = opt_job.compiled_program().data.count("RZZ")
    print(f"level {level}: two-qubit gates = {two_qubit_gates}, counts {counts}")

Same answer at every level, since the LE emulator is noiseless. The difference is in the
compiled program: level 0 keeps all three `CX` gates as three `RZZ` gates, and level 2 cuts
them down to the single one the Bell pair needs.

That matters off the noiseless emulator. Two-qubit gates are the main source of error on
trapped-ion hardware, and the main driver of a hardware job's cost in HQCs (Quantinuum's
billing unit), so fewer of them means both cleaner results and a cheaper run.

The compiler validates the level. An out-of-range value fails the job before anything runs,
and the reason is on the job's status, in `status_message`, which is the first place to look
when a job fails:

In [ ]:
bad_job = device.run(redundant, shots=10, runtime_options={"optimisation_level": 5})
bad_job.wait_for_final_state(timeout=600, poll_interval=5)

status = bad_job.status()
print(status)
print(status.status_message)

## Noiseless vs. noisy

Now an eight-qubit GHZ state, on the noiseless H2 emulator and on the H2 emulator with the
hardware noise model. Ideally every shot reads all zeros or all ones; anything else is an
error.

In [ ]:
noisy_device = provider.get_device("quantinuum:quantinuum:sim:h2-emulator")

ghz8 = ghz(8)
ideal_job = device.run(ghz8, shots=200)
noisy_job = noisy_device.run(ghz8, shots=200)

for name, ghz_job in [("H2-1LE", ideal_job), ("H2-Emulator", noisy_job)]:
    counts = get_counts(ghz_job)
    good = counts.get("0" * 8, 0) + counts.get("1" * 8, 0)
    print(f"{name:12} {good / sum(counts.values()):.1%} of shots in 00000000 or 11111111")

The noiseless emulator only ever returns the two ideal outcomes. On the noisy emulator a few
shots leak into other bitstrings. The emulator models the device's gate, measurement,
state-preparation, and idling errors, so the more gates and qubits a circuit uses, the more
of these you see.

Develop and debug on an LE target, then check on the matching Emulator target how the circuit
holds up under realistic noise.

In [ ]:
from qbraid.visualization import plot_histogram

plot_histogram(get_counts(noisy_job))

## Mid-circuit measurement and reset

Trapped-ion qubits can be measured and reset **partway through** a circuit, then reused.
Here qubit 0 is flipped to 1, measured, reset to 0, and reused to build a Bell pair with
qubit 1:

In [ ]:
from qiskit import ClassicalRegister, QuantumRegister

qubits = QuantumRegister(2, "q")
bits = ClassicalRegister(3, "c")
reuse = QuantumCircuit(qubits, bits)

reuse.x(0)
reuse.measure(0, 0)  # always 1
reuse.reset(0)
reuse.h(0)
reuse.cx(0, 1)
reuse.measure(0, 1)
reuse.measure(1, 2)

reuse.draw()

In [ ]:
get_counts(device.run(reuse, shots=200))

The first measurement (rightmost bit) is always `1`, and the Bell pair after the reset gives
`00` or `11` on the other two bits, so only `001` and `111` appear.

## Running several circuits

The Quantinuum devices don't support **batch jobs** (several circuits in one job), so
`as_batch=True` is rejected before anything is submitted:

In [ ]:
print("batch support:", device.profile.batch_job_support)

try:
    device.run([ghz(2), ghz(3)], shots=100, as_batch=True)
except ValueError as err:
    print("ValueError:", err)

Pass a list of circuits **without** `as_batch` and each one is submitted as a **separate job**;
`run()` returns a list of jobs, in order. They run in parallel on NEXUS, so a sweep takes
roughly as long as its slowest circuit, not the sum.

A natural use is a parameter sweep. Rotate one qubit by $\theta$ with `RY` and the
probability of measuring 1 is $\sin^2(\theta/2)$:

In [ ]:
import numpy as np

angles = np.linspace(0, np.pi, 6)

sweep = []
for theta in angles:
    circuit = QuantumCircuit(1)
    circuit.ry(theta, 0)
    circuit.measure_all()
    sweep.append(circuit)

sweep_jobs = device.run(sweep, shots=200)
print(len(sweep_jobs), "jobs")

sweep_counts = [get_counts(sweep_job) for sweep_job in sweep_jobs]

In [ ]:
import matplotlib.pyplot as plt

measured = [counts.get("1", 0) / sum(counts.values()) for counts in sweep_counts]

fine = np.linspace(0, np.pi, 200)
plt.plot(fine, np.sin(fine / 2) ** 2, label=r"$\sin^2(\theta/2)$")
plt.plot(angles, measured, "o", label="H2-1LE, one job per angle")
plt.xlabel(r"$\theta$")
plt.ylabel("P(1)")
plt.legend()
plt.show()

Each point is its own job with its own ID, status, and independent sampling.

## Summary

- Four Quantinuum emulators are available through `QbraidProvider`, free of charge: `h1-1le` and `h2-1le` are noiseless, `h1-emulator` and `h2-emulator` model the hardware's noise.
- NEXUS compiles every circuit to the trapped-ion native gates; `job.compiled_program()` shows what ran.
- The one runtime option is `optimisation_level` (`0` to `3`, default `1`), which trades compile effort for fewer two-qubit gates. Other options are ignored, and there is no `seed`.
- Mid-circuit measurement and qubit reset work as on the hardware.
- There are no batch jobs: a list of circuits runs as separate, parallel jobs.
- When a job fails, `job.status().status_message` says why.

<div class="alert alert-block alert-info">
<b>Copyright Notice:</b> 
    All rights reserved © [2026] qBraid. This notebook is part of the qBraid-Lab-Demo repository.
The qBraid-Lab-Demo is licensed under the Apache License, Version 2.0.
You may obtain a copy of the License at <https://www.apache.org/licenses/LICENSE-2.0>.
Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
</div>